In [9]:
#read all csv
import os
import pandas as pd

# Define the path to the data directory
data_dir = "../data/raw/csv/ttc_subway_delay"

# List all CSV files in the data directory
csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

years_list = ['2023', '2024', '2025', '2026']
# Read each CSV file into a DataFrame based on year in filename and store in a list
dfs = []
for file in csv_files:
    # Extract year from filename (assuming format like "fafd_afdaf_2020.csv")   
    year = file.split('.')[0][-4:]  # Get the last 4 characters (the year)
    # Only process files for years in the specified list
    if str(year) in years_list:
        df = pd.read_csv(os.path.join(data_dir, file))
        df['year'] = year  # Add a column for the year
        dfs.append(df)
        print(f"Loaded {file}")

# Concatenate all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)
print("All CSV files have been combined into a single DataFrame.")

#drop id column
if 'id' in combined_df.columns:
    combined_df = combined_df.drop(columns=['id'])
    print("Dropped 'id' column from the combined DataFrame.")

#delete duplicate rows and print count of duplicate rows
duplicate_rows = combined_df[combined_df.duplicated(keep=False)]
print("Count of duplicate rows:", len(duplicate_rows))
combined_df = combined_df.drop_duplicates()
print("Removed duplicate rows from the combined DataFrame.")

#rename all the columns in combined_df as per dictionary
column_mapping = {
    'Date': 'date',
    'Time': 'time',
    'Day': 'day',
    'Station': 'station',
    'Code': 'code',
    'Min Delay': 'delayed_minutes',
    'Min Gap': 'gap_minutes',
    'Bound': 'bound',
    'Line': 'line',
    'Vehicle': 'vehicle'
}

# Rename columns in combined_df
combined_df = combined_df.rename(columns=column_mapping)

# Display the first few rows of the combined DataFrame
print(combined_df.head())

#write in data/processed/combined_data.csv# Define the path to the output file
output_file = "../data/processed/ttc_subway_delay_data_combined.csv"
# Save the combined DataFrame to a new CSV file
combined_df.to_csv(output_file, index=False)
print(f"Combined data has been saved to {output_file}.")

Loaded ttc-subway-delay-data-2026.csv
Loaded ttc-subway-delay-data-2024.csv
Loaded ttc-subway-delay-data-2025.csv
Loaded ttc-subway-delay-data-2023.csv
All CSV files have been combined into a single DataFrame.
Dropped 'id' column from the combined DataFrame.
Count of duplicate rows: 170
Removed duplicate rows from the combined DataFrame.
         date   time       day                station  code  delayed_minutes  \
0  2026-01-01  02:09  Thursday       SHEPPARD STATION  SUAP                0   
1  2026-01-01  02:28  Thursday        KIPLING STATION  MUIS                0   
2  2026-01-01  03:19  Thursday  SHEPPARD WEST STATION  MUIS                0   
3  2026-01-01  03:44  Thursday            VMC STATION  MUIS                0   
4  2026-01-01  03:53  Thursday        KIPLING STATION  MUIS                0   

   gap_minutes bound line  vehicle  year  
0            0   NaN   YU        0  2026  
1            0     E   BD     5179  2026  
2            0   NaN   YU        0  2026  
3      

In [11]:
#read ttc station mapping file in processed folder
mapping_ttc_station_file = "../data/processed/mapping_ttc_station.csv"
mapping_ttc_station_df = pd.read_csv(mapping_ttc_station_file)

#print count of duplicate rows
duplicate_rows_mapping = mapping_ttc_station_df[mapping_ttc_station_df.duplicated(keep=False)]
print("Count of duplicate rows in the mapping DataFrame:", len(duplicate_rows_mapping))

#find duplicate station names in the mapping file
duplicate_station_names = mapping_ttc_station_df[mapping_ttc_station_df.duplicated(subset=['station'], keep=False)]
print("Duplicate station names in the mapping DataFrame:")
print(duplicate_station_names)


#read ttc delay code mapping file in processed folder
mapping_ttc_delay_code_file = "../data/processed/mapping_ttc_delay_code.csv"
mapping_ttc_delay_code_df = pd.read_csv(mapping_ttc_delay_code_file)

#print count of duplicate rows
duplicate_rows_mapping = mapping_ttc_delay_code_df[mapping_ttc_delay_code_df.duplicated(keep=False)]
print("Count of duplicate rows in the mapping DataFrame:", len(duplicate_rows_mapping))

#find duplicate station names in the mapping file
duplicate_station_names = mapping_ttc_delay_code_df[mapping_ttc_delay_code_df.duplicated(subset=['delay_code'], keep=False)]
print("Duplicate station names in the mapping DataFrame:")
print(duplicate_station_names)





Count of duplicate rows in the mapping DataFrame: 0
Duplicate station names in the mapping DataFrame:
Empty DataFrame
Columns: [station, station_clean, row_count, mapped_station, score, include_station, remarks]
Index: []
Count of duplicate rows in the mapping DataFrame: 0
Duplicate station names in the mapping DataFrame:
Empty DataFrame
Columns: [delay_code, row_count, mapped_delay_code, fuzzy_score, include_code, remarks]
Index: []


In [12]:
#clean data for line and bound columns
def canonical_line(x):
    if pd.isna(x):
        return "UNKNOWN"

    s = str(x).upper().replace(" ", "")

    present = set()
    if "YU" in s:
        present.add("YU")
    if "BD" in s:
        present.add("BD")
    if "SHP" in s:
        present.add("SHP")
    if "SRT" in s:
        present.add("SRT")

    order = ["YU", "BD", "SHP", "SRT"]
    tokens = [t for t in order if t in present]

    return "/".join(tokens) if tokens else "UNKNOWN"


def canonical_bound(x):
    if pd.isna(x):
        return "UNKNOWN"

    s = str(x).strip().upper()

    mapping = {
        "N": "N", "NB": "N", "NORTH": "N",
        "S": "S", "SB": "S", "SOUTH": "S",
        "E": "E", "EB": "E", "EAST": "E",
        "W": "W", "WB": "W", "WEST": "W",
    }
    return mapping.get(s, "UNKNOWN")


combined_df["line_clean"] = combined_df["line"].apply(canonical_line)
combined_df["bound_clean"] = combined_df["bound"].apply(canonical_bound)

In [13]:
#create one big table by merging combined_df with mapping__ttc_station_df and mapping__ttc_delay_code_df
# Merge combined_df with mapping__ttc_station_df on 'station' column and rename columns
merged_df = pd.merge(combined_df, mapping_ttc_station_df, on='station', how='left') \
    .rename(columns={'score': 'station_score', 'remarks': 'station_remarks'}) \
    .drop(columns=['row_count', 'station_clean'])
# Merge merged_df with mapping__ttc_delay_code_df on 'code' column
merged_df = pd.merge(merged_df, mapping_ttc_delay_code_df, left_on='code', right_on='delay_code', how='left') \
    .rename(columns={'score': 'delay_code_score', 'remarks': 'delay_code_remarks', 'description': 'delay_code_description'}) \
    .drop(columns=['row_count'])
print("Merged DataFrame has been created by combining the combined DataFrame with the mapping DataFrames.")
#print the columns of merged_df
print("Columns in the merged DataFrame:")
print(merged_df.columns.tolist())

#write merged_df to obt csv file in processed folder
output_file_merged = "../data/processed/ttc_subway_delay_data_obt.csv"
merged_df.to_csv(output_file_merged, index=False)
print(f"Merged data has been saved to {output_file_merged}.")

Merged DataFrame has been created by combining the combined DataFrame with the mapping DataFrames.
Columns in the merged DataFrame:
['date', 'time', 'day', 'station', 'code', 'delayed_minutes', 'gap_minutes', 'bound', 'line', 'vehicle', 'year', 'line_clean', 'bound_clean', 'mapped_station', 'station_score', 'include_station', 'station_remarks', 'delay_code', 'mapped_delay_code', 'fuzzy_score', 'include_code', 'delay_code_remarks']
Merged data has been saved to ../data/processed/ttc_subway_delay_data_obt.csv.


In [14]:
print(merged_df.shape)

(77521, 22)


In [15]:
# read data from ttc_subway_delay_data_combined.csv into data frame and display the first few rows
import pandas as pd

# Define the path to the input file
input_file = "../data/processed/ttc_subway_delay_data_combined.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(input_file)

# Display the first few rows of the DataFrame
print(df.head())


         date   time       day                station  code  delayed_minutes  \
0  2026-01-01  02:09  Thursday       SHEPPARD STATION  SUAP                0   
1  2026-01-01  02:28  Thursday        KIPLING STATION  MUIS                0   
2  2026-01-01  03:19  Thursday  SHEPPARD WEST STATION  MUIS                0   
3  2026-01-01  03:44  Thursday            VMC STATION  MUIS                0   
4  2026-01-01  03:53  Thursday        KIPLING STATION  MUIS                0   

   gap_minutes bound line  vehicle  year  
0            0   NaN   YU        0  2026  
1            0     E   BD     5179  2026  
2            0   NaN   YU        0  2026  
3            0   NaN   YU        0  2026  
4            0   NaN   BD        0  2026  


In [17]:
#get distinct codes with count from the data frame
distinct_codes = df['code'].value_counts()
print(distinct_codes)

code
SUDP     10131
MUIS      7945
SUO       6732
MUPAA     4656
MUIRS     4047
         ...  
MRFS         1
PRSW         1
PRW          1
ERBO         1
ERTL         1
Name: count, Length: 186, dtype: int64
